# Biomarker Analysis Pipeline (v4)

Two cohorts, each with two propensity score models and two analysis tracks.

### Cohorts
- **Cohort 1** (`cohort1`) — First-line ICI vs all never-ICI, unmatched
- **Cohort 2** (`cohort2`) — Lines 1-3, 1:1 matched on (cancer_type, line_category)

### Propensity score models (trained within each cohort)
- **covariates_only** — elastic net CV LR on demographics + cancer type + line
- **covariates_plus_embeddings** — elastic net CV LR on covariates + text embeddings

### Analysis tracks (using cohort-specific PS without adjustment)
- **Track 1** — ICI-only, prognostic: `S(t) ~ base_vars + line_dummies + marker`
  - **unweighted** + **ATE** (1/ps generalizability weights)
- **Track 2** — Full cohort, predictive interaction: `S(t) ~ base_vars + line_dummies + marker + ICI + marker x ICI`
  - **noIPTW** + **ATE** weights

### Stages
1. **Cohort construction** — `build_line_matched_cohort.py` (produces both cohorts)
2. **Propensity scores** — `ICI_LRs.py` (loops over both cohorts)
3. **IPTW datasets** — `generate_IPTW_df.py` (loops over all cohort × ps_model combinations)
4. **Cox models** — `run_IPTW_analysis.py` (loops over all cohort × ps_model combinations)
5. **Compile results** — `compile_IPTW_results.py` (aggregates significant hits)

Each script is notebook-ready — just `%run` each one in sequence. All looping over cohorts and PS models is handled internally.

In [ ]:
import os
import sys

SCRIPT_DIR = os.path.dirname(os.path.abspath('__file__'))
# Ensure script directory is on the path for imports
if SCRIPT_DIR not in sys.path:
    sys.path.insert(0, SCRIPT_DIR)

## Stage 1: Cohort Construction

Build both cohorts:
- **Cohort 1**: First-line ICI vs all never-ICI, unmatched
- **Cohort 2**: Lines 1-3, 1:1 matched on (cancer_type, line_category)

In [ ]:
%run build_line_matched_cohort.py

## Stage 2: Propensity Score Generation

Train covariates-only and covariates+embeddings propensity models within each cohort.
Loops over both cohorts automatically.

In [ ]:
%run ICI_LRs.py

## Stage 3: IPTW Dataset Generation

Build IPTW datasets for each {cohort, ps_model} combination.
Loops over all 4 combinations automatically.

In [ ]:
%run generate_IPTW_df.py

## Stage 4: Cox Model Analysis

Run Track 1 (ICI-only) and Track 2 (full-cohort interaction) for each {cohort, PS model} combination.
Loops over all 4 combinations automatically.

In [ ]:
%run run_IPTW_analysis.py

## Stage 5: Compile Results

Aggregate significant hits across all cohorts and specifications.

In [ ]:
%run compile_IPTW_results.py